# N173 · Hard序列、字符串与设计综合

**目标：** 独立推导状态或候选支配并比较两种解法。

**先修：** N029, N056, N107, N170, N171。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最短覆盖；单调队列；前缀；区间合并；状态设计；适用条件。

**配套讲解来源：** [同名逐章意见](../../comment/173_hard_sequences_strings_capstone.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章是序列与字符串方向的综合 capstone，方法论主线是：**Hard 题的难点通常不是代码更长，而是"好几个小前提必须同时成立"**。你要么独立推导出正确的状态设计（窗口计数的口径、DP 的分割步长），要么识别出候选之间的支配关系（单调队列删掉无用状态），并且能说清每个解法的适用条件。

三道承载题各演示一个前提被破坏的后果，先各给一个具体例子：

- **最小覆盖子串（LC 76）**：`s='ADOBECODEBANC'`，`t='ABC'`，答案 `'BANC'`。关键前提：要数的是**缺少的字符实例数**，不是"缺少哪几种字符"。如果 `t='AAB'`，窗口里只有一个 A 是不够的——两个 A 实例都得覆盖到。
- **带负数的最短子数组（LC 862）**：`nums=[2,-1,2]`，`k=3`，答案是 3（整段和恰为 3）。关键前提：数组含负数时，"窗口变长和就变大"不再成立，普通滑动窗口失效，必须换前缀和 + 单调队列。
- **合并石头（LC 1000）**：`stones=[3,2,4,1]`，`k=2`（每次合并相邻 2 堆），最少代价 20；同一数组换 `k=3`，答案是 -1——因为每次合并把堆数减少 k-1，从 4 堆到 1 堆要减 3，而 3 不能被 2 整除，**根本合不成一堆**，先判可达性再谈代价。

## 2. 基础知识：先读懂这些词

- **重数（multiplicity）**：目标串里每个字符需要的**个数**，而不是"出现没出现"。最小覆盖窗口的计数口径必须是实例数：`t='AAB'` 需要窗口里至少有两个 A。口径选错，代码短不了多少，但一定错。
- **缺额计数（missing）**：`min_window` 用一个整数 `missing` 汇总"还缺多少个字符实例"，它等于零当且仅当每个目标字符的重数都被满足。它是滑动窗口状态的核心不变量。
- **前缀支配（单调队列）**：对最短子数组问题，以 j 结尾的最优左端点是"前缀值最小、下标尽量大"的候选。若某个旧下标 i 的前缀 `prefixes[i] >= prefixes[j']`（j' 更新），那 i 作为左端点既远又不占便宜，被 j' **支配**，可以从队尾安全删掉。队里只剩"越旧前缀越小"的候选。
- **可达性判定**：合并石头里 `(n-1)%(k-1)==0` 是能把 n 堆合成 1 堆的充要条件。每合并一次堆数恰好减少 k-1，所以总减少量 n-1 必须是 k-1 的整数倍。先检查这个条件，不满足直接返回 -1，省掉整个 DP。
- **区间 DP**：`dp[left][right]` 表示"把区间 `[left,right]` 按规则合并到所能达到的最少合法堆数时的最小成本"。先算短区间，再拼长区间。分割点步长是 k-1 而不是 1，这是本题状态设计的灵魂。
- **加总重量的物理含义**：只有当一个区间最终能压成一堆时，才把该区间的总重量加进成本——它正是"把这一堆 k 份合成一份"那一刻的代价。说不清这一步的物理含义，DP 就只是抄来的表格。
- **适用条件**：每个解法成立的前提清单（窗口法要求非负数组、单调队列法兼容负数、区间 DP 要求可达性成立）。本章三个接口都叫 `*_audited`，提醒你"实现 + 论证"要成对出现。
- **强参照与规模纪律**：每个快速实现都配一个"慢但按定义算"的暴力参照（枚举子串、枚举子数组、记忆化枚举合并过程），参照只在小输入上运行；大规模输入只冒烟跑正式实现。

## 3. 思路推导：从小例子开始

- **Step 1（窗口）：初始化计数。** `required=Counter(t)` 记录每个字符还差几个实例，`missing=len(t)` 是总缺额。手算 `s='ADOBECODEBANC'`、`t='ABC'`：初始缺 A、B、C 各一个，missing=3。
- **Step 2（窗口）：右端扩张。** 右指针扫过字符时，如果该字符还缺（`required[char]>0`）就把 missing 减一；然后无论是否需要都执行 `required[char]-=1`（多出来的会记成负数，表示盈余）。扫到 `ADOBEC` 时 A、B、C 各命中一次，missing 归零。
- **Step 3（窗口）：左端收缩。** missing==0 时窗口合法，先尝试记录更短的答案，再从左边吐出一个字符：把它在 required 里加回来，如果加回后变成正数（>0），说明吐掉的是"刚需"，missing 加一、停止收缩。手算：合法窗口 `ADOBEC` 吐掉 A 后缺额重现，左端停在 1；此后右端继续扫，直到 r=12 的 C 再次凑齐，收缩左端依次吐掉 O、D、E、B（都是盈余，missing 保持 0），窗口缩到 `BANC`，长度 3，更新为最优；再吐 A 时缺额重现，结束。答案 `BANC`。
- **Step 3.5（窗口）：换个重复目标再验一次口径。** 把目标换成 `t='AAB'`、`s='AAAB'`：初始缺 A 两个、B 一个，missing=3。右端扫过第一个 A（缺额 3→2）、第二个 A（2→1）、第三个 A（盈余，`required['A']` 记成 -1）、B（1→0），窗口合法。收缩时吐掉第一个 A：`required['A']` 从 -1 加回 0，不是正数，missing 不动——盈余可以丢；再吐第二个 A 时 `required['A']` 变 1，缺额重现，停止。最终答案 `AAB`（长度 3）。对比一个数"种类"的错误实现：`t='AA'`、`s='AA'` 时它看到窗口 `'A'` 里 A 这种字符出现过就判合法，会错误地输出 `'A'`；而按实例数计数的正确答案显然是 `'AA'`——这正是练习 2 要你构造的致病输入。

- **Step 4（最短子数组）：先看普通窗口为什么死。** 若数组非负，窗口和随右端单调增，收缩安全；`[2,-1,2]` 里中间的 -1 让窗口先变大后变小，"固定右端缩左端"的口诀不成立。所以改用前缀和：区间 `[i,j)` 的和是 `prefixes[j]-prefixes[i]`，条件 `>=k` 变成"找最大的 i 使 `prefixes[i] <= prefixes[j]-k`"。
- **Step 5（最短子数组）：单调队列两删。** 前缀数组是 `[0,2,1,3]`。对每个 j：**队头删**——若 `prefixes[j]-prefixes[queue[0]]>=k`，队头是可行的左端点，而且越往后 j 越大、子数组越长，所以这个队头以后不可能更优，弹出并顺手更新 `best=min(best,j-弹出下标)`；**队尾删**——若 `prefixes[queue[-1]]>=prefixes[j]`，旧下标被 j 支配（j 更新、前缀不更大），删掉。手算：j=3（前缀 3）时队头 0（前缀 0），3-0>=3 ✓，弹出得长度 3-0=3，这就是答案；中间 j=2（前缀 1）曾把 j=1（前缀 2）从队尾删掉，因为前者更小更近。
- **Step 6（合并石头）：先判可达性。** `n=4`、`k=3`：`(4-1)%(3-1)=1≠0`，永远合不成一堆，返回 -1。`k=2` 时余数为 0，可以继续。
- **Step 7（合并石头）：区间 DP 一格一格手算（k=2，stones=[3,2,4,1]）。** 前缀和 `prefix=[0,3,5,9,10]`。长度 2 的区间：`dp[0][1]=0+区间和=5`（合并 3,2 花 5），`dp[1][2]=6`（合并 2,4），`dp[2][3]=5`（合并 4,1）。长度 3：`dp[0][2]=min(dp[0][0]+dp[1][2], dp[0][1]+dp[2][2])+9=min(6,5)+9=14`，`dp[1][3]=min(dp[1][1]+dp[2][3], dp[1][2]+dp[3][3])+7=min(5,6)+7=12`。长度 4：`dp[0][3]=min(dp[0][0]+dp[1][3], dp[0][1]+dp[2][3], dp[0][2]+dp[3][3])+10=min(12,10,14)+10=20`。答案 20——与 “运行示例”部分 表里的 `interval merge` 行一致。
- **Step 8（合并石头）：看懂两个设计点。** 分割点 `middle` 按 `k-1` 步长穷举（k=2 时就是每个位置），保证左段 `[left,middle]` 恰好能压成一堆；`(length-1)%(k-1)==0` 时整个区间能压成一堆，此时才加区间总和——那正是最后一次合并 k 堆的代价（物理含义）。
- **Step 9：小对拍 + 大冒烟分开跑。** “自动测试”部分 用指数级/立方级的暴力参照只在小输入上对拍，规模冒烟（长 20000 的输入）只调这三个 O(n)/O(n²) 的正式实现，绝不碰暴力参照。

## 4. 核心代码：min_window_audited

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def min_window_audited(s, t):
    if not t:
        return ''
    required = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, char in enumerate(s):
        if required[char] > 0:
            missing -= 1
        required[char] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            required[old] += 1
            left += 1
            if required[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]
```

### 逐段读懂代码

### 1. 最小覆盖窗口 `min_window_audited`

```python
def min_window_audited(s, t):
    if not t:
        return ''
    required = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, char in enumerate(s):
        if required[char] > 0:
            missing -= 1
        required[char] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            required[old] += 1
            left += 1
            if required[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]
```

- `required` 记每个字符的**剩余需求实例数**，`missing=len(t)` 是总缺额——这就是"数实例不数种类"的落点；`Counter` 查不存在的键返回 0，所以 `required[char]>0` 对无关字符自然不触发。
- 右端每进一个字符：还缺时 missing 减一，然后一律 `required[char]-=1`（盈余记成负数）。
- `while missing==0` 内部先更新答案（`best` 存 `(起点,终点)` 且终点为排他端点，长度就是 `best[1]-best[0]`），再吐出左端字符：`required[old]+=1` 后若 `>0` 说明吐掉了刚需，missing 加一、循环自动停止。
- 返回时无解给空串。`if not t` 的空目标处理成返回空串，和暴力参照的口径一致。
- 值得注意的写法细节：`required` 里"盈余"用负数表示（比如窗口里多出三个 A 就是 -3），收缩时 `required[old]+=1` 只是让盈余少一个，只有加完之后 `>0` 才意味着吐掉了刚需。一个整数同时承担"还缺几个/多出几个"两种语义，避免了再开一个"窗口内计数"字典的双份状态。

### 2. 最短子数组 `shortest_subarray_audited`

```python
def shortest_subarray_audited(nums, k):
    if k <= 0:
        raise ValueError('Positive threshold required')
    prefixes = [0]
    for value in nums:
        prefixes.append(prefixes[-1] + value)
    queue = deque()
    best = len(nums) + 1
    for j, value in enumerate(prefixes):
        while queue and value - prefixes[queue[0]] >= k:
            best = min(best, j - queue.popleft())
        while queue and prefixes[queue[-1]] >= value:
            queue.pop()
        queue.append(j)
    return -1 if best > len(nums) else best
```

- 先建含空前缀的前缀和数组（`prefixes[0]=0`），保证以 0 开头的子数组也能表示成两前缀之差；`best` 初值设为 `len(nums)+1` 这个不可能的长度当哨兵。
- 队头删：`value-prefixes[queue[0]]>=k` 成立说明队头下标可作为左端点；因为 j 递增，这个队头今后只会配出更长的区间，弹出不会丢最优——`best=min(best,j-queue.popleft())` 一行完成"结算+删除"。
- 队尾删：`prefixes[queue[-1]]>=value` 说明旧队尾被当前 j 支配（更旧、前缀还不更小），删掉。两条规则合起来，队列里下标递增、前缀严格递减。
- 每个下标最多进队一次、出队一次，整体线性；`return -1 if best>len(nums) else best` 用哨兵值判断无解。

### 3. 合并石头 `merge_stones_audited`

（对应程序见下方分段实现与完整代码。）

- 前三行是输入契约：k 至少为 2（k=1 的"合并"无意义且会让 `(n-1)%(k-1)` 除零）、石头重量非负。`n<=1` 已经是一堆，代价 0。
- `if (n-1)%(k-1):return -1` 是可达性判断：整除才有解，不整除连 DP 都不用跑。
- `prefix` 前缀和让任意区间总重量 O(1) 可查。DP 表按区间长度从小到大填：`dp[left][right]` 取分割点 `middle` 从 `left` 到 `right-1`、**步长 k-1** 的所有候选的最小值——步长保证左段长度 `(middle-left+1)` 满足 `(len-1)%(k-1)==0`，即左段自己能压成一堆。
- `if (length-1)%(k-1)==0: dp[left][right]+=prefix[right+1]-prefix[left]`：整段能压成一堆时，加上的正是"最后把 k 堆合成 1 堆"这一步的代价，等于区间总重量——这就是推导里说的物理含义。区间长度不满足整除时，这个区间最多只能压到 `(L-1) mod (k-1)+1` 堆，不能加总重量，只留作更大区间的半成品。

## 5. 分段实现：按顺序运行

**本章接口：** `min_window_audited(s, t)`、`shortest_subarray_audited(nums, k)`、`merge_stones_audited(stones, k)`

### 导入本章使用的库

In [1]:
from collections import Counter, deque

### min_window_audited

In [2]:
def min_window_audited(s, t):
    if not t:
        return ''
    required = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, char in enumerate(s):
        if required[char] > 0:
            missing -= 1
        required[char] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            required[old] += 1
            left += 1
            if required[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]

### shortest_subarray_audited

In [3]:
def shortest_subarray_audited(nums, k):
    if k <= 0:
        raise ValueError('Positive threshold required')
    prefixes = [0]
    for value in nums:
        prefixes.append(prefixes[-1] + value)
    queue = deque()
    best = len(nums) + 1
    for j, value in enumerate(prefixes):
        while queue and value - prefixes[queue[0]] >= k:
            best = min(best, j - queue.popleft())
        while queue and prefixes[queue[-1]] >= value:
            queue.pop()
        queue.append(j)
    return -1 if best > len(nums) else best

### merge_stones_audited

In [4]:
def merge_stones_audited(stones, k):
    if k < 2:
        raise ValueError('At least two adjacent piles per merge')
    if any((value < 0 for value in stones)):
        raise ValueError('Nonnegative pile weights required')
    n = len(stones)
    if n <= 1:
        return 0
    if (n - 1) % (k - 1):
        return -1
    prefix = [0]
    for value in stones:
        prefix.append(prefix[-1] + value)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for left in range(n - length + 1):
            right = left + length - 1
            dp[left][right] = min((dp[left][middle] + dp[middle + 1][right] for middle in range(left, right, k - 1)))
            if (length - 1) % (k - 1) == 0:
                dp[left][right] += prefix[right + 1] - prefix[left]
    return dp[0][-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='ADOBECODEBANC';t='ABC'
show_table(['mechanism','input','result'],[
 ['multiplicity window',(s,t),min_window_audited(s,t)],
 ['prefix dominance',([2,-1,2],3),shortest_subarray_audited([2,-1,2],3)],
 ['interval merge',([3,2,4,1],2),merge_stones_audited([3,2,4,1],2)],
 ['unreachable remainder',([3,2,4,1],3),merge_stones_audited([3,2,4,1],3)]])

mechanism,input,result
multiplicity window,"('ADOBECODEBANC', 'ABC')",BANC
prefix dominance,"([2, -1, 2], 3)",3
interval merge,"([3, 2, 4, 1], 2)",20
unreachable remainder,"([3, 2, 4, 1], 3)",-1


## 7. 正确性、适用条件与复杂度

窗口缺额为零等价于每个目标字符重数都满足。单调队列删除的是被较新、更小前缀支配的旧状态。石头区间长度 L 可压缩到 `(L−1) mod (k−1)+1` 堆；分割令左段可压一堆，穷举其最后分界覆盖所有合法结构。只有余数为零时还能把 k 堆合成一堆，成本恰是区间总和。

**代价：** 前两个算法时间 O(n+|t|)、O(n)，空间分别 O(字符种类)、O(n)。合并石头 O(n³/(k−1)) 时间（上界 O(n³)）、O(n²) 空间。独立穷举与规模冒烟分开。

### 展开理解

**窗口为什么对。** 不变量是：`missing==0` 当且仅当窗口内每个目标字符的实例数都不低于需求重数。右端每进一个字符，缺额最多减一；左端吐掉刚需字符时缺额恰好加一。所以每个"第一次合法"的窗口都会被发现，每个合法窗口又都被收缩到极限——最短答案必然被枚举到。注意这一切建立在"我们数的是实例数"上，`t='AAB'` 这类重复目标被口径天然覆盖。

**单调队列为什么对。** 对固定的 j，最优左端点是满足 `prefixes[i]<=prefixes[j]-k` 的**最大 i**。队头删不丢解：被弹出的 i 配当前 j 已满足条件，而以后的 j' 更大、`j'-i` 更长，不可能更优。队尾删不丢解：被删的旧下标 i 的前缀不小于新下标 j 的前缀，任何未来 j' 若能和 i 配对（差 >= k），也一定能和 j 配对（差不减）且 `j'-j < j'-i` 更短。两条合起来，留下的候选两两无可替代。

**区间 DP 为什么对。** 可达性 `(n-1)%(k-1)==0` 来自"每次合并堆数减 k-1"的守恒。DP 的正确性分两层：任何合法的合并过程都可以看成"先把区间分成左右两段各自压成若干堆、最后一段式合并"，而枚举步长 k-1 的分割点恰好覆盖所有"左段可压成一堆"的结构；区间能整体压成一堆时最后一步的代价恰是区间总和，加一次、只加一次。三方（暴力递归、区间 DP、判定条件）在测试里对齐。

**代价。** 窗口法：左右指针各走一遍 s，时间 O(n+|t|)，计数字典只存目标字符，空间 O(字符种类)。最短子数组：每个下标进出队列至多一次，时间 O(n)，队列和前缀数组空间 O(n)。合并石头：三层循环（长度 × 起点 × 分割点），分割点按步长 k-1 抽稀，时间 O(n³/(k-1))（上界 O(n³)），空间 O(n²)。拿具体规模说：两万长的字符串窗口法线性扫过；30 堆石头、k=2 的 DP 也就三万级格子，冒烟测试瞬间跑完——而指数级的暴力参照只敢在长度 7 以内运行。

**测试代价的对应关系。** “自动测试”部分 里三组参照的规模选择和上述代价一一对应：窗口暴力是 O(n³) 级（每个子串再数一遍字符），所以穷举域限制在长度 6 的 `AB` 串；子数组暴力是 O(n²)，随机长度不超过 8；石头暴力是"指数级状态 × 每状态 k 个选择"，n 压到 7 以内。参照永远比正式实现慢至少一个量级，这正是它能当裁判的原因——两个实现以不同速度算同一个问题，答案必须一致。

**适用条件清单。** 窗口法要求"计数口径是实例数"；前缀支配队列兼容负数但要求阈值 k 为正（代码里 `k<=0` 直接抛错，因为含任意负数的"和至少为 k"在 k<=0 时定义会退化）；区间 DP 要求可达性 `(n-1)%(k-1)==0`、k≥2、重量非负。三份前提都写成了函数开头的 `ValueError`，契约违约是显式失败而不是静默错误答案。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分五组：

1. **窗口穷举对拍**：对长度 0–6 的全部 `AB` 串 × 目标 `['','A','AA','AB','AAB']`，与"枚举所有子串取最短"的 `brute_window` 比对。`AA`、`AAB` 这两个目标专门测**重数**——只数种类的错误实现对 `t='AA'` 会输出更短但不合法的窗口。
2. **窗口冒烟**：`min_window_audited('ADOBECODEBANC','ABC')=='BANC'`，经典正常值用例，答案可手算（第三节推过）。
3. **最短子数组：边界 + 随机对拍**：`[-2,-1]`、k=1 返回 -1，验证"全负无解"的边界；再用固定种子 `Random(173)` 生成 250 组随机数组（长 0–8、值域 -5..7）和 k=1..14，与 O(n²) 枚举所有子数组的表达式对拍，覆盖了负数导致普通窗口失效的场景。注意穷举域刻意包含空数组（`rng.randrange(9)` 可以取 0）和"正负混合"的值域，前者测空输入返回 -1 还是 0 的口径（k>=1 且无元素时无解，应为 -1），后者正是普通窗口的失效区。
3.5. **最短子数组的边界用例再展开一下**：`[-2,-1]`、k=1 这条之所以重要，是因为两个前缀 `[0,-2,-3]` 两两之差最大才 0，永远够不到 1——无解分支 `-1` 被精确覆盖；而随机对拍里 `default=-1` 的生成式写法（`min(..., default=-1)`）同时承担了"有解取最短、无解给 -1"两种期望，确保两个分支都被几百组随机数据反复踩到。
4. **合并石头：记忆化暴力对拍**：`brute_merge` 用 `@cache` 递归枚举"每次选连续 k 堆合并"的所有可能，状态是石头元组、代价是每次合并的 k 堆之和；递归到底长度 ≤1 返回 0，所有选法都不可达则停在 `float('inf')`（测试里转成 -1）。对 n=0..7、k=2,3,4 各随机 8 组对拍——k=3、n=4 这类不可达情形由 `inf` 分支对齐 -1，`@cache` 让指数级搜索在长度 7 以内勉强可跑。另有 `[3,2,4,1],2==20` 和 `[3,2,4,1],3==-1` 两条冒烟，分别对应第三节手算的 20 和可达性判定。
5. **规模冒烟**：两万长的 `'x'*20000+'AAB'` 窗口答案 `'AAB'`（顺带验证无关字符不干扰计数——两万个 'x' 把 `required['x']` 压到 -20000 但从不影响 missing）、两万个 1 组成的数组上 `k=100` 的答案是 100（单调队列在纯正数上退化成一次线性扫描，但结果必须仍对）、30 堆石头 k=2 代价为正。这组**只调正式实现**，绝不调指数级/立方级参照——notebook 注释里明确写了这条纪律。

In [6]:
from functools import cache

from itertools import product

from random import Random

def brute_window(s, t):
    if not t:
        return ''
    need = Counter(t)
    best = ''
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            found = Counter(s[i:j])
            if all((found[x] >= count for x, count in need.items())):
                if not best or j - i < len(best):
                    best = s[i:j]
                break
    return best

for n in range(7):
    for chars in product('AB', repeat=n):
        s = ''.join(chars)
        for t in ['', 'A', 'AA', 'AB', 'AAB']:
            assert min_window_audited(s, t) == brute_window(s, t)

assert min_window_audited('ADOBECODEBANC', 'ABC') == 'BANC'

assert shortest_subarray_audited([-2, -1], 1) == -1

rng = Random(173)

for _ in range(250):
    nums = [rng.randrange(-5, 8) for _ in range(rng.randrange(9))]
    k = rng.randrange(1, 15)
    expected = min((j - i for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) >= k), default=-1)
    assert shortest_subarray_audited(nums, k) == expected

@cache
def brute_merge(state, k):
    if len(state) <= 1:
        return 0
    answer = float('inf')
    for i in range(len(state) - k + 1):
        merged = sum(state[i:i + k])
        cost = brute_merge(state[:i] + (merged,) + state[i + k:], k)
        answer = min(answer, merged + cost)
    return answer

for n in range(8):
    for k in [2, 3, 4]:
        for _ in range(8):
            stones = tuple((rng.randrange(1, 6) for _ in range(n)))
            expected = brute_merge(stones, k)
            expected = -1 if expected == float('inf') else expected
            assert merge_stones_audited(stones, k) == expected

assert merge_stones_audited([3, 2, 4, 1], 2) == 20

assert merge_stones_audited([3, 2, 4, 1], 3) == -1

assert min_window_audited('x' * 20000 + 'AAB', 'AAB') == 'AAB'

assert shortest_subarray_audited([1] * 20000, 100) == 100

assert merge_stones_audited([1] * 30, 2) > 0

print('N173: 所有本章断言通过')

N173: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 完成一题不看标签的变体。

**练习 2：** 为每题提供一个导致常见错误的最小输入。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不看标签做变体）**：拿本章一题换个皮：比如把"最短覆盖窗口"改成"最长无冗余覆盖"、把合并石头的 k 换个值重算。写清楚输入输出与边界（空 t、k 超过堆数、全负数组），先手算一个 5–6 个元素的例子，再用本章接口或独立暴力解验证。核心是逼自己重新走一遍"状态/支配关系"的推导，而不是回忆题目标签。
- **练习 2（最小致病输入）**：三题各找一个"让常见错误实现翻车"的最小输入。窗口题：`t='AAB'`、`s='AB'`——数种类的实现会误判覆盖成立。子数组题：`nums=[2,-1,2]`、k=3——普通双指针会漏掉跨负数的整段。石头题：`stones=[3,2,4,1]`、k=3——不检查可达性的实现会返回一个错误代价而不是 -1。每个例子写明错误实现的输出与正确输出，并指出错在哪一步推理；用手算展示那一步，再用本章接口或暴力解印证。

写致病输入有个通用套路：先列出正确实现的每条前提（实例数口径、非负性、整除条件），然后逐条问"违反它的最小输入长什么样"。三个最小反例都只有 3–4 个元素，正因为小，你才能在纸上把错误实现每一步的中间状态写出来、指认出错的精确位置——这也是 N170 反例工作坊方法在 Hard 题上的复用。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter, deque

def min_window_audited(s, t):
    if not t:
        return ''
    required = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, char in enumerate(s):
        if required[char] > 0:
            missing -= 1
        required[char] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            required[old] += 1
            left += 1
            if required[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]

def shortest_subarray_audited(nums, k):
    if k <= 0:
        raise ValueError('Positive threshold required')
    prefixes = [0]
    for value in nums:
        prefixes.append(prefixes[-1] + value)
    queue = deque()
    best = len(nums) + 1
    for j, value in enumerate(prefixes):
        while queue and value - prefixes[queue[0]] >= k:
            best = min(best, j - queue.popleft())
        while queue and prefixes[queue[-1]] >= value:
            queue.pop()
        queue.append(j)
    return -1 if best > len(nums) else best

def merge_stones_audited(stones, k):
    if k < 2:
        raise ValueError('At least two adjacent piles per merge')
    if any((value < 0 for value in stones)):
        raise ValueError('Nonnegative pile weights required')
    n = len(stones)
    if n <= 1:
        return 0
    if (n - 1) % (k - 1):
        return -1
    prefix = [0]
    for value in stones:
        prefix.append(prefix[-1] + value)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for left in range(n - length + 1):
            right = left + length - 1
            dp[left][right] = min((dp[left][middle] + dp[middle + 1][right] for middle in range(left, right, k - 1)))
            if (length - 1) % (k - 1) == 0:
                dp[left][right] += prefix[right + 1] - prefix[left]
    return dp[0][-1]

# 示例与回归测试
from functools import cache

from itertools import product

from random import Random

def brute_window(s, t):
    if not t:
        return ''
    need = Counter(t)
    best = ''
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            found = Counter(s[i:j])
            if all((found[x] >= count for x, count in need.items())):
                if not best or j - i < len(best):
                    best = s[i:j]
                break
    return best

for n in range(7):
    for chars in product('AB', repeat=n):
        s = ''.join(chars)
        for t in ['', 'A', 'AA', 'AB', 'AAB']:
            assert min_window_audited(s, t) == brute_window(s, t)

assert min_window_audited('ADOBECODEBANC', 'ABC') == 'BANC'

assert shortest_subarray_audited([-2, -1], 1) == -1

rng = Random(173)

for _ in range(250):
    nums = [rng.randrange(-5, 8) for _ in range(rng.randrange(9))]
    k = rng.randrange(1, 15)
    expected = min((j - i for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) >= k), default=-1)
    assert shortest_subarray_audited(nums, k) == expected

@cache
def brute_merge(state, k):
    if len(state) <= 1:
        return 0
    answer = float('inf')
    for i in range(len(state) - k + 1):
        merged = sum(state[i:i + k])
        cost = brute_merge(state[:i] + (merged,) + state[i + k:], k)
        answer = min(answer, merged + cost)
    return answer

for n in range(8):
    for k in [2, 3, 4]:
        for _ in range(8):
            stones = tuple((rng.randrange(1, 6) for _ in range(n)))
            expected = brute_merge(stones, k)
            expected = -1 if expected == float('inf') else expected
            assert merge_stones_audited(stones, k) == expected

assert merge_stones_audited([3, 2, 4, 1], 2) == 20

assert merge_stones_audited([3, 2, 4, 1], 3) == -1

assert min_window_audited('x' * 20000 + 'AAB', 'AAB') == 'AAB'

assert shortest_subarray_audited([1] * 20000, 100) == 100

assert merge_stones_audited([1] * 30, 2) > 0

print('N173: 所有本章断言通过')

N173: 所有本章断言通过


## 11. 代表题与迁移

- **76. Minimum Window Substring（comparison）**：练"重数缺额计数 + 双指针收缩"的状态设计，`min_window_audited` 是完整实现，`t` 含重复字符时尤其能体现口径的重要性。
- **862. Shortest Subarray with Sum at Least K（comparison）**：练"前缀和 + 前缀支配单调队列"，体会负数如何破坏普通窗口的前提，`shortest_subarray_audited` 即其实现。
- **1000. Minimum Cost to Merge Stones（comparison）**：练"可达性判定 + 步长 k-1 的区间 DP"，`merge_stones_audited` 覆盖了 `(n-1)%(k-1)` 判 -1 和 DP 两个层面。

三道题合起来看，迁移路径是同一个：**先找到被破坏的前提（重数口径、非负性、整除可达性），再决定换哪个机制来补**。窗口题换计数口径、子数组题换前缀支配、石头题换 DP 状态与步长——Hard 的"难"大多落在这个选择上，而不是落在代码量上。

notebook 结尾照例提醒：这些题号用于知识映射，本章实现遵循教学契约，不要把教学实例当成官方题的完整答案。

**题面入口：** [862](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/) · [1000](https://leetcode.com/problems/minimum-cost-to-merge-stones/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。